# Visualizing CLP Standard Errors

This short notebook demonstrates the new CLP standard-error functionality in pyglotaran.

It runs one optimization with `compute_clp_standard_error=True` and visualizes both:
- the fitted conditionally linear parameters (`clp`)
- their standard errors (`clp_standard_error`)

In [ ]:
from dataclasses import fields
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from glotaran.optimization.optimize import optimize
from glotaran.testing.simulated_data.sequential_spectral_decay import SCHEME

In [ ]:
scheme_field_names = {field.name for field in fields(SCHEME)}

required_fields = {
    "compute_clp_standard_error",
    "clp_standard_error_finite_difference_relative_step",
}

if not required_fields.issubset(scheme_field_names):
    missing = sorted(required_fields - scheme_field_names)
    raise RuntimeError(
        "This notebook requires a pyglotaran version with CLP standard-error support. "
        f"Missing Scheme fields: {missing}.\n"
        "Install pyglotaran from a version/branch that includes CLP standard errors, "
        "then restart the kernel and rerun."
    )

scheme = replace(
    SCHEME,
    compute_clp_standard_error=True,
    clp_standard_error_finite_difference_relative_step=1e-6,
    maximum_number_function_evaluations=20,
)

result = optimize(scheme, raise_exception=True)
dataset_name = next(iter(result.data))
result_data = result.data[dataset_name]

print(f"Optimization success: {result.success}")
print(f"Dataset: {dataset_name}")
print("Result-level CLP-SE method:", result.clp_standard_error_method)
print("Variables in result dataset:", list(result_data.data_vars))

In [ ]:
assert "clp_standard_error" in result_data, (
    "No 'clp_standard_error' found. Ensure compute_clp_standard_error=True and "
    "the residual function supports CLP-SE."
)

clp = result_data["clp"]
clp_se = result_data["clp_standard_error"]

display(clp)  # noqa: F821
display(clp_se)  # noqa: F821

In [ ]:
# Select one CLP label and visualize value + standard error over (time, spectral)
clp_label = clp.coords["clp_label"].values[0]
clp_map = clp.sel(clp_label=clp_label)
clp_se_map = clp_se.sel(clp_label=clp_label)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

clp_map.plot(ax=axes[0])
axes[0].set_title(f"CLP values ({clp_label})")

clp_se_map.plot(ax=axes[1])
axes[1].set_title(f"CLP standard error ({clp_label})")

plt.show()

In [ ]:
# Relative uncertainty can be helpful for interpretation
relative_se = np.abs(clp_se_map / np.where(np.abs(clp_map) > 1e-12, clp_map, np.nan))

plt.figure(figsize=(6, 4))
relative_se.plot()
plt.title(f"Relative CLP uncertainty |SE / CLP| ({clp_label})")
plt.show()

## Notes

- `clp_standard_error` uses a linear covariance term plus finite-difference propagation of nonlinear parameter uncertainty.
- It is currently intended for residual functions that support this pathway (for example variable projection).
- Areas with larger values in the SE map indicate less certain CLP estimates.